# Museum | PGSR

Enable a Kaggle GPU and Internet. Attach the original
[Museum.mp4](../../datasets/tanks_and_temples/museum/Museum.mp4) as a private Dataset.
The video checksum is pinned. Fresh SfM requires FFmpeg, ffprobe and a Linux CUDA
COLMAP build that passes the repository's exclusive CASPAR bundle-adjustment probe.

Build the source-only capsule from the Windows repository root.

~~~powershell
python -m scripts.kaggle.museum_handoff source --destination outputs/tanks_and_temples/Museum/kaggle_handoff/source
~~~

Attach the resulting private Kaggle dataset containing museum_source.bundle. Set
SOURCE_SHA256 from source_manifest.json. For a matched comparison, attach the same
prepared Museum dataset to each notebook. For museum_scene.bundle, set SCENE_ARCHIVE_SHA256
and SCENE_RECEIPT_SHA256 from scene_manifest.json. For an extracted copy, use SCENE_INPUT
with SCENE_RECEIPT_SHA256.
The prepared input can come from the local handoff CLI or from a successful notebook
scene export. Leave both scene input options unset only to produce a fresh GPU-strict Museum SfM.

For same-method recovery, attach the saved result, reuse its scene, set PREVIOUS_OUTPUT
and enter the earlier checkpoint SHA-256. The full native checkpoint identity is checked.

## 1. Configure Kaggle inputs

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import zipfile
from uuid import uuid4

METHOD = 'B2'
SOURCE_SHA256 = ''  # SHA-256 of your trusted museum_source.bundle
COLMAP_BIN = None  # optional absolute path to a CUDA/CASPAR Linux executable
SCENE_INPUT = None  # optional path to a trusted saved Museum output
SCENE_RECEIPT_SHA256 = ''
SCENE_ARCHIVE_SHA256 = ''  # optional hash of attached museum_scene.bundle
PREVIOUS_OUTPUT = None  # optional path to your same-method Museum output
PREVIOUS_CHECKPOINT_SHA256 = ''
PROFILE = 'research_quality'  # smoke, low_memory, research_quality
LOG_INTERVAL = 100
SESSION_TARGET = 30000  # absolute session target, with the optimizer schedule kept at 30k
VIDEO_SHA256 = 'f407eca1d989587c5c350fce42156c235b6e17dedc08c1013fa739d1088d83f2'
SESSION_STARTED = time.monotonic()
SESSION_BUDGET_SECONDS = 3.5 * 3600
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

WORK = Path(f'/tmp/museum-{METHOD.lower()}/project')
OUTPUT = Path(f'/kaggle/working/museum_{METHOD.lower()}')
MUSEUM_ROOT = Path('datasets/tanks_and_temples/museum')
VIDEO_PATH = MUSEUM_ROOT / 'Museum.mp4'
SFM_ROOT = Path('outputs/tanks_and_temples/Museum/sfm_colmap/video_strict_v1')
PREPARED_ROOT = MUSEUM_ROOT / 'prepared/video_v1'

## 2. Verify and unpack the project source

In [ ]:
def digest(path):
    with path.open('rb') as source:
        hasher = hashlib.sha256()
        for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
            hasher.update(block)
        return hasher.hexdigest()


def mounted(name):
    matches = sorted(path for path in Path('/kaggle/input').rglob(name) if path.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one attached {name}; found {matches}')
    return matches[0]


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def save_json(path, value):
    path.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')


def run_visible(command):
    with subprocess.Popen(
        list(map(str, command)), cwd=WORK, env=ENV, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    ) as child:
        for line in child.stdout:
            print(line, end='', flush=True)
        if child.wait():
            raise subprocess.CalledProcessError(child.returncode, child.args)


archive = mounted('museum_source.bundle')
if digest(archive) != SOURCE_SHA256.strip().lower():
    raise ValueError('Set SOURCE_SHA256 to the known hash of your trusted source archive')
if WORK.exists() or OUTPUT.exists():
    raise FileExistsError('Use a fresh session to preserve existing outputs')
with zipfile.ZipFile(archive) as capsule:
    for entry in capsule.infolist():
        relative = Path(entry.filename)
        if (
            relative.is_absolute() or '..' in relative.parts or chr(92) in entry.filename
            or not relative.parts or relative.parts[0] not in {'src', 'scripts', 'configs'}
            or ((entry.external_attr >> 16) & 0o170000) == 0o120000
        ):
            raise ValueError(f'Unexpected capsule member {entry.filename!r}')
    WORK.mkdir(parents=True)
    capsule.extractall(WORK)

OUTPUT.mkdir(parents=True)
ENV = {**os.environ, 'PYTHONUNBUFFERED': '1'}
if COLMAP_BIN is not None:
    ENV['COLMAP_BIN'] = str(COLMAP_BIN)
run_visible(['nvidia-smi'])

## 3. Prepare the Museum scene with verified GPU SfM

In [ ]:
import yaml


def load_yaml(relative):
    return yaml.safe_load((WORK / relative).read_text(encoding='utf-8'))


def verified_sfm(path):
    metrics = read_json(path)
    if (
        metrics['status'] != 'SUCCESS' or metrics['source_video_sha256'] != VIDEO_SHA256
        or metrics['gpu']['ba']['verified'] is not True
        or metrics['gpu']['ba']['backend'] != 'CASPAR'
    ):
        raise ValueError('Missing successful GPU BA evidence for Museum.mp4')


if SCENE_INPUT is None and SCENE_ARCHIVE_SHA256:
    archive = mounted('museum_scene.bundle')
    if digest(archive) != SCENE_ARCHIVE_SHA256.strip().lower():
        raise ValueError('Prepared Museum archive checksum mismatch')
    SCENE_INPUT = WORK / 'mounted_museum_scene'
    with zipfile.ZipFile(archive) as packed:
        for entry in packed.infolist():
            relative = Path(entry.filename)
            if (
                relative.is_absolute() or '..' in relative.parts or chr(92) in entry.filename
                or not relative.parts or relative.parts[0] not in {'scene', 'scene.json', 'metrics.json'}
                or ((entry.external_attr >> 16) & 0o170000) == 0o120000
            ):
                raise ValueError(f'Unexpected scene archive member {entry.filename!r}')
        SCENE_INPUT.mkdir()
        packed.extractall(SCENE_INPUT)

if SCENE_INPUT is None:
    if not shutil.which(ENV.get('COLMAP_BIN', 'colmap')):
        raise RuntimeError('BLOCKED_GPU_BA: set COLMAP_BIN to a CUDA/CASPAR Linux executable')
    video = mounted('Museum.mp4')
    staged_video = WORK / VIDEO_PATH
    if shutil.disk_usage(WORK).free <= video.stat().st_size:
        raise RuntimeError('Insufficient workspace space for the Museum video')
    staged_video.parent.mkdir(parents=True)
    shutil.copyfile(video, staged_video)
    if digest(staged_video) != VIDEO_SHA256:
        raise ValueError('Museum.mp4 checksum differs from the repository video')

    sfm = load_yaml('configs/sfm_museum_strict.yaml')
    sfm['input'].update(scene='Museum', video=VIDEO_PATH.as_posix())
    sfm['paths'] = {
        'extracted_frames': (MUSEUM_ROOT / 'preprocessing/extracted_frames/video_v0').as_posix(),
        'selected_frames': (MUSEUM_ROOT / 'preprocessing/selected_frames/video_strict_v1').as_posix(),
        'output': SFM_ROOT.as_posix(),
    }
    # Use the minimum required by the subsequent prepared-data configuration.
    sfm['sfm']['min_registered_images'] = 3
    sfm_config = WORK / 'configs/sfm_museum_kaggle.yaml'
    save_json(sfm_config, sfm)  # JSON is valid YAML.
    command = [sys.executable, '-u', '-m', 'src.pipeline', '--config', sfm_config]
    try:
        run_visible([*command, '--stage', 'preflight'])
        try:
            run_visible([*command, '--stage', 'all', '--resume'])
        except subprocess.CalledProcessError:
            actual = WORK / SFM_ROOT / 'metrics.json'
            if not actual.is_file() or read_json(actual)['status'] != 'STOP_MAPPING_INVALID':
                raise
    finally:
        for name in ('environment.json', 'gpu_report.json', 'metrics.json', 'colmap_state.json'):
            artifact = WORK / SFM_ROOT / name
            if artifact.is_file():
                shutil.copyfile(artifact, OUTPUT / ('sfm_pipeline_' + name))

    qualified_scene = WORK / 'museum_scene_handoff'
    run_visible([sys.executable, '-u', '-m', 'scripts.kaggle.museum_handoff', 'scene',
                 '--sfm-config', sfm_config, '--destination', qualified_scene])
    shutil.copyfile(qualified_scene / 'metrics.json', OUTPUT / 'metrics.json')
    verified_sfm(OUTPUT / 'metrics.json')
    [metadata_file] = (WORK / PREPARED_ROOT).glob('*/metadata.json')
    DATASET = metadata_file.parent
else:
    source = Path(SCENE_INPUT)
    if digest(source / 'scene.json') != SCENE_RECEIPT_SHA256.strip().lower():
        raise ValueError('Use the known scene receipt hash from the trusted Museum run')
    receipt = read_json(source / 'scene.json')
    if receipt['video_sha256'] != VIDEO_SHA256:
        raise ValueError('Saved scene belongs to a different video')
    if digest(source / 'metrics.json') != receipt['sfm_metrics_sha256']:
        raise ValueError('Saved SfM metrics checksum mismatch')
    verified_sfm(source / 'metrics.json')
    dataset_id = receipt['dataset_id']
    if not dataset_id.startswith('museum-video-v1-') or Path(dataset_id).name != dataset_id:
        raise ValueError('Unexpected Museum dataset ID')
    prepared = source / 'scene' / dataset_id
    if digest(prepared / 'integrity.json') != receipt['integrity_sha256']:
        raise ValueError('Prepared scene integrity manifest checksum mismatch')
    DATASET = WORK / PREPARED_ROOT / dataset_id
    shutil.copytree(prepared, DATASET)
    shutil.copyfile(source / 'metrics.json', OUTPUT / 'metrics.json')

metadata = read_json(DATASET / 'metadata.json')
if (
    metadata['dataset_id'] != DATASET.name or metadata['source']['mode'] != 'video'
    or metadata['source']['manifest']['video_sha256'] != VIDEO_SHA256
):
    raise ValueError('Prepared dataset provenance does not match Museum.mp4')
run_visible([
    sys.executable, '-u', '-c',
    "import sys; from pathlib import Path; from src.data.loader import verify_dataset; "
    "report = verify_dataset(Path(sys.argv[1])); print(report); sys.exit(report['status'] != 'SUCCESS')",
    DATASET,
])
shutil.copytree(DATASET, OUTPUT / 'scene' / DATASET.name)
receipt_path = OUTPUT / 'scene.json'
save_json(receipt_path, {
    'dataset_id': DATASET.name,
    'video_sha256': VIDEO_SHA256,
    'split_hash': metadata['split_hash'],
    'integrity_sha256': digest(DATASET / 'integrity.json'),
    'sfm_metrics_sha256': digest(OUTPUT / 'metrics.json'),
})
print('Scene receipt SHA-256:', digest(receipt_path), flush=True)

## 4. Build the isolated CUDA environment

In [ ]:
run_visible(['bash', WORK / 'scripts/kaggle_pgsr/setup_linux.sh'])
PY = WORK / '.tools/pgsr_env/bin/python'
if not PY.is_file():
    raise FileNotFoundError(PY)
ENV['CUDA_HOME'] = str(PY.parents[1])

## 5. Configure the fixed-split Museum training run

In [ ]:
if not 1 <= SESSION_TARGET <= 30000 or LOG_INTERVAL < 1:
    raise ValueError('Use a session target in 1..30000 and a positive log interval')
if PROFILE not in {'smoke', 'low_memory', 'research_quality'}:
    raise ValueError('Unknown Museum profile')
profile = load_yaml(f'configs/training/museum/pgsr_{PROFILE}.yaml')
SESSION_TARGET = min(SESSION_TARGET, profile.get('iterations', 30000))
output_root = Path('outputs/tanks_and_temples/Museum/research') / 'B2_PGSR'
profile.update(
    dataset=DATASET.relative_to(WORK).as_posix(),
    output_root=output_root.as_posix(),
    log_interval=LOG_INTERVAL,
)
profile['session_seconds'] = int(SESSION_BUDGET_SECONDS)
CONFIG = WORK / 'configs/training/museum_pgsr_kaggle.yaml'
save_json(CONFIG, profile)
if PREVIOUS_OUTPUT is None:
    RUN_ID = f"{profile['experiment']}-{uuid4().hex[:10]}"
else:
    RUN_ID = read_json(Path(PREVIOUS_OUTPUT) / 'training_input.json')['run_id']
    if Path(RUN_ID).name != RUN_ID or not RUN_ID.startswith(profile['experiment'] + '-'):
        raise ValueError('Prior Museum run ID does not match this profile')
RUN = WORK / output_root / RUN_ID
TRAIN_COMMAND = [
    str(PY), '-u', '-m', 'src.pgsr.runner', '--config', str(CONFIG),
    '--run', str(RUN), '--stop-after', str(SESSION_TARGET),
]
save_json(OUTPUT / 'training_input.json', {
    'method': METHOD,
    'run_id': RUN_ID,
    'video_sha256': VIDEO_SHA256,
    'source_capsule_sha256': SOURCE_SHA256.strip().lower(),
    'scene_receipt_sha256': digest(OUTPUT / 'scene.json'),
    'profile': profile,
})
if PREVIOUS_OUTPUT is not None:
    previous = Path(PREVIOUS_OUTPUT)
    if read_json(previous / 'training_input.json') != read_json(OUTPUT / 'training_input.json'):
        raise ValueError('Resume requires identical method, source capsule, scene and training profile')
    prior_run = previous / 'run'
    pointer = read_json(prior_run / 'checkpoints/latest.json')
    name = pointer['path']
    if Path(name).name != name or not 0 < pointer['iteration'] < SESSION_TARGET:
        raise ValueError('Resume target must exceed the saved checkpoint iteration')
    checkpoint_path = prior_run / 'checkpoints' / name
    if pointer['sha256'] != PREVIOUS_CHECKPOINT_SHA256.strip().lower() or digest(checkpoint_path) != pointer['sha256']:
        raise ValueError('Set PREVIOUS_CHECKPOINT_SHA256 to the saved hash from your own run')
    shutil.copytree(prior_run, RUN)
    log = RUN / 'training.jsonl'
    if log.is_file():
        # Keep sampled observations covered by the resumed checkpoint in the active history.
        original = log.read_text(encoding='utf-8')
        kept = [
            line for line in original.splitlines(keepends=True)
            if line.strip() and json.loads(line)['iteration'] <= pointer['iteration']
        ]
        if len(kept) != len(original.splitlines()):
            with (RUN / 'training_prior_sessions.jsonl').open('a', encoding='utf-8') as history:
                history.write(original.rstrip('\n') + '\n')
            log.write_text(''.join(kept), encoding='utf-8')
    TRAIN_COMMAND.extend(['--resume'])


def export_run():
    if not RUN.is_dir():
        print('Training artifacts unavailable.')
        return
    pointer_path = RUN / 'checkpoints/latest.json'
    latest = None
    if pointer_path.is_file():
        pointer = read_json(pointer_path)
        latest = pointer['path']
        if Path(latest).name != latest or digest(RUN / 'checkpoints' / latest) != pointer['sha256']:
            raise ValueError('Published checkpoint checksum mismatch')
        print('Checkpoint SHA-256:', pointer['sha256'], flush=True)
    saved = OUTPUT / 'run'
    shutil.copytree(
        RUN, saved,
        ignore=lambda _, names: [name for name in names if name.endswith('.pth') and name != latest],
    )
    log = saved / 'training.jsonl'
    if log.is_file():
        with log.open(encoding='utf-8') as records:
            last = None
            for line in records:
                if line.strip():
                    last = line
        if last:
            sample = json.loads(last)
            print('Latest sampled iteration:', sample['iteration'], flush=True)
            print(json.dumps(sample.get('losses', {'total': sample['loss']}), indent=2), flush=True)
        run_visible([sys.executable, '-u', '-m', 'scripts.kaggle.loss_report',
                     log, OUTPUT / 'losses'])
    else:
        print('Loss unavailable. No training samples were recorded.', flush=True)
    print('Saved run:', saved, flush=True)

## 6. Train and print losses

The runner streams loss samples at LOG_INTERVAL. The exported training.jsonl contains all recorded raw and weighted terms. The notebook prints the latest sample and saves the latest verified checkpoint, when available.

In [ ]:
if SESSION_BUDGET_SECONDS - (time.monotonic() - SESSION_STARTED) <= 20 * 60:
    raise RuntimeError('Session has insufficient time to train and export a checkpoint')
try:
    run_visible(TRAIN_COMMAND)
finally:
    export_run()

## 7. Evaluate completed training

Save a Kaggle version with outputs to retain the scene, checkpoint, and loss history.

In [ ]:
result_path = RUN / 'training_result.json'
if result_path.is_file() and read_json(result_path)['status'] == 'SUCCESS':
    try:
        run_visible([PY, '-u', '-m', 'src.pgsr.runner', '--config', CONFIG, '--run', RUN,
                     '--stage', 'evaluate'])
        # The PNG scorer reads the pinned B0 SSIM source as text for the parity check.
        b0_reference = WORK / 'external/gaussian-splatting'
        b0_revision = '414b553ef1d3032ea634b7b7a1d121173f36592c'
        if not b0_reference.exists():
            run_visible(['git', 'clone', '--no-checkout',
                         'https://github.com/graphdeco-inria/gaussian-splatting.git', b0_reference])
            run_visible(['git', '-C', b0_reference, 'checkout', '--detach', b0_revision])
        actual = subprocess.check_output(['git', '-C', str(b0_reference), 'rev-parse', 'HEAD'],
                                         text=True).strip()
        if actual != b0_revision:
            raise ValueError('Unexpected B0 SSIM reference revision')
        run_visible([PY, '-u', 'scripts/pgsr/score_png.py', '--run', RUN])
    finally:
        for name in ('evaluation',):
            path = RUN / name
            if path.is_dir():
                shutil.copytree(path, OUTPUT / 'run' / name)
            elif path.is_file():
                shutil.copyfile(path, OUTPUT / 'run' / name)
else:
    print('Held-out evaluation requires completed training.')